# Demo 4 — STII causal verification + ACDC minimal circuits (fairness)

From correlation to causation: STII quantifies group effects; ACDC distills minimal circuits. This notebook reproduces the Demo 4 pipeline headlessly available via [`python/demo4_causal.py`](python/demo4_causal.py:1).

In [ ]:
import sys, os, json, platform
import numpy as np
import torch, transformers, sklearn

print({
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "sklearn": sklearn.__version__,
})
print("Reminder: build py_nsi first if not installed: cd py_nsi && maturin develop --release")

# Make local modules importable (namespace package "python" at repo root)
sys.path.append(os.path.abspath("."))

In [ ]:
from python.utils.config import load_yaml  # [def load_yaml()](python/utils/config.py:1)
CFG_PATH = "configs/demo4_causal.yaml"
cfg = load_yaml(CFG_PATH)
cfg

In [ ]:
from python.datasets.loans_bias import generate_loans_dataset  # [def generate_loans_dataset()](python/datasets/loans_bias.py:1)

ds = cfg["dataset"]
texts, labels, genders = generate_loans_dataset(
    n_samples=int(ds["n_samples"]),
    seed=int(ds["seed"]),
    bias_strength=float(ds["bias_strength"]),
    noise=float(ds["noise"]),
)
import numpy as np
labels_np = np.asarray(labels, dtype=np.int32)
genders_np = np.asarray(genders, dtype=np.int32)
print("N=", len(texts))
print("label rate denied=1:", labels_np.mean())
print("gender female=1 rate:", genders_np.mean())
print("sample text:", texts[0] if texts else "")

In [ ]:
from python.activations.extract import get_model_and_tokenizer, capture_layer_activations  # [def capture_layer_activations()](python/activations/extract.py:23)

model_name = cfg["model_name"]
layer_index = int(cfg["layer_index"])
model, tokenizer = get_model_and_tokenizer(model_name)
acts = capture_layer_activations(model, tokenizer, texts, layer_index=layer_index)
acts.shape

In [ ]:
from python.ensemble.intersection import build_pyensemble  # [def build_pyensemble()](python/ensemble/intersection.py:37)
ens = cfg["ensemble"]
os.environ["PY_NSI_INPUT_DIM"] = str(int(acts.shape[1]))
ensemble = build_pyensemble(
    feature_dim=int(ens["feature_dim"]),
    top_k=int(ens["top_k"]),
    seeds=[int(s) for s in ens["seeds"]],
)
ensemble

In [ ]:
from python.hypergraph.pipeline import build_hypergraph_with_nodes  # [def build_hypergraph_with_nodes()](python/hypergraph/pipeline.py:28)
spk = cfg["spike"]
store, X_edge_bool, edge_keys, nodes_by_sample_bool, node_keys = build_hypergraph_with_nodes(
    ensemble=ensemble,
    acts=acts,
    labels=labels_np,
    t_start=float(spk["t_start"]),
    delta_t=float(spk["delta_t"]),
    min_sigmoid=float(spk["min_sigmoid"]),
    gse_window=float(spk["gse_window"]),
)
X_edge_bool.shape, nodes_by_sample_bool.shape, len(edge_keys), len(node_keys)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_node = nodes_by_sample_bool.astype(np.float32)
y = labels_np.astype(np.int32)
X_tr, X_te, y_tr, y_te = train_test_split(X_node, y, test_size=0.2, random_state=int(ds["seed"]), stratify=y)
clf = LogisticRegression(solver="liblinear", random_state=int(ds["seed"]), max_iter=1000)
clf.fit(X_tr, y_tr)
acc_base = accuracy_score(y_te, clf.predict(X_te))
acc_base

In [ ]:
from python.stii.compute import compute_stii_for_hyperedge  # [def compute_stii_for_hyperedge()](python/stii/compute.py:1)

node_to_col = {int(nid): j for j, (nid,) in enumerate(node_keys)}
stii_cfg = cfg["stii"]
stii_vals = {}
count = 0
for ek in edge_keys:
    if len(ek) <= 3:
        try:
            stii_vals[ek] = float(compute_stii_for_hyperedge(
                store=store, edge_key=ek, node_to_col=node_to_col,
                X_base=X_node, y=y, logreg_model=clf, max_order_k=int(stii_cfg.get("max_order_k", 2)),
            ))
            count += 1
        except Exception:
            stii_vals[ek] = 0.0
sorted_edges = sorted(((ek, v) for ek, v in stii_vals.items()), key=lambda t: t[1], reverse=True)
sorted_edges[:10]

In [ ]:
from python.acdc.prune import acdc_minimal_circuit  # [def acdc_minimal_circuit()](python/acdc/prune.py:1)
X_edge = X_edge_bool.astype(np.float32)
acdc_cfg = cfg["acdc"]
acdc_res = acdc_minimal_circuit(
    edge_keys=edge_keys, stii=stii_vals, X_edge=X_edge, y=y,
    tolerance_drop=float(acdc_cfg["tolerance_drop"]), max_edges=int(acdc_cfg["max_edges"]), seed=int(ds["seed"])
)
acdc_res["base_acc"], acdc_res["final_acc"], len(acdc_res["kept_edges"]), len(acdc_res["removed_edges"])

In [ ]:
from python.metrics.fairness import gender_concept_probs, report_bias_presence  # [def gender_concept_probs()](python/metrics/fairness.py:1)
node_gender = gender_concept_probs(nodes_by_sample=nodes_by_sample_bool, genders=genders_np)
edge_to_nodes = {ek: [int(n) for n in ek] for ek in edge_keys}
fairness = report_bias_presence(
    minimal_edges=acdc_res["kept_edges"],
    edge_to_nodes=edge_to_nodes,
    node_gender_probs=node_gender,
    node_keys=node_keys,
    threshold=0.6,
)
fairness

In [ ]:
from python.utils.artifacts import create_run_dir, dump_json, dump_yaml  # [def create_run_dir()](python/utils/artifacts.py:1)
out_cfg = cfg["outputs"]
run_dir = create_run_dir(base_dir=out_cfg["base_dir"], run_tag=out_cfg.get("run_tag"))
dump_yaml(cfg, os.path.join(run_dir, "config.yaml"))
store.export_hif(os.path.join(run_dir, "hypergraph_stii.hif.json"))
stii_list = [
    {"edge": [int(x) for x in ek], "stii": float(stii_vals.get(ek, 0.0))}
    for ek in edge_keys if len(ek) <= 3
]
dump_json({"values": stii_list, "computed_count": int(len(stii_list)), "total_edges": int(X_edge.shape[1])}, os.path.join(run_dir, "stii_values.json"))
dump_json({
    "kept_edges": [[int(x) for x in ek] for ek in acdc_res.get("kept_edges", [])],
    "removed_edges": [[int(x) for x in ek] for ek in acdc_res.get("removed_edges", [])],
    "base_acc": float(acdc_res.get("base_acc", 0.0)),
    "final_acc": float(acdc_res.get("final_acc", 0.0)),
    "tolerance_drop": float(acdc_cfg["tolerance_drop"]),
    "max_edges": int(acdc_cfg["max_edges"]),
}, os.path.join(run_dir, "acdc_minimal_circuit.json"))
dump_json(fairness, os.path.join(run_dir, "fairness_report.json"))
print("Artifacts written to", run_dir)
print("Acceptance mapping: HIF+STII, stii_values.json, acdc_minimal_circuit.json, fairness_report.json saved.")